# C04 — Object Detection: IoU, NMS, and YOLO

> **Audience**: PhD students · **Framework**: PyTorch + NumPy · **Dataset**: synthetic + COCO via ultralytics

Object detection predicts **both** what objects are present and **where** they are
(bounding box coordinates + class label).

**What this notebook builds from scratch**
1. Bounding box representations and conversions
2. Intersection over Union (IoU) — the universal overlap metric
3. Non-Maximum Suppression (NMS) — collapsing duplicate detections
4. Anchor box assignment — how detectors match predictions to ground truth
5. YOLO output decoding — turning network output into bounding boxes

**Modern inference**
Section 6 shows how to run YOLOv8 (3 lines of code) — the contrast highlights
what the framework abstracts away from the maths we implement manually.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)
torch.manual_seed(42)

# 1) Bounding Box Representations

Two representations are used throughout detection literature:

| Format | Fields | Typical use |
|--------|--------|-------------|
| `[x1, y1, x2, y2]` | top-left corner + bottom-right corner | IoU, NMS, evaluation |
| `[cx, cy, w, h]`   | centre + width/height | YOLO prediction head, anchor encoding |

Converting between them is a constant source of off-by-one bugs.
Always annotate which format a variable holds.

In [ ]:
def xyxy_to_cxcywh(boxes: torch.Tensor) -> torch.Tensor:
    """
    Converts corner format to centre format.

    Args:
        boxes : (..., 4) tensor in [x1, y1, x2, y2] format

    Returns:
        Tensor (..., 4) in [cx, cy, w, h] format
    """
    # (num_boxes, 4) → (num_boxes, 4)
    x1, y1, x2, y2 = boxes.unbind(dim=-1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w  = x2 - x1
    h  = y2 - y1
    return torch.stack([cx, cy, w, h], dim=-1)


def cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    """
    Converts centre format to corner format.

    Args:
        boxes : (..., 4) tensor in [cx, cy, w, h] format

    Returns:
        Tensor (..., 4) in [x1, y1, x2, y2] format
    """
    # (num_boxes, 4) → (num_boxes, 4)
    cx, cy, w, h = boxes.unbind(dim=-1)
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return torch.stack([x1, y1, x2, y2], dim=-1)


# ── Round-trip verification ────────────────────────────────────────────────────
boxes_xyxy = torch.tensor([
    [10., 20., 50., 60.],   # w=40, h=40, cx=30, cy=40
    [100., 150., 200., 250.],
])

boxes_cxcywh  = xyxy_to_cxcywh(boxes_xyxy)
boxes_back    = cxcywh_to_xyxy(boxes_cxcywh)

print("Original [x1,y1,x2,y2]:")
print(boxes_xyxy)
print("Converted [cx,cy,w,h]:")
print(boxes_cxcywh)
print(f"Round-trip matches: {torch.allclose(boxes_xyxy, boxes_back)}")

# 2) Intersection over Union (IoU)

IoU is the fundamental metric for bounding box overlap:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{intersection area}}{\text{area}(A) + \text{area}(B) - \text{intersection area}}$$

**Properties**
- IoU = 0: boxes do not overlap at all
- IoU = 1: boxes are identical
- IoU > 0.5: commonly used as a "true positive" threshold in mAP evaluation

**Vectorised implementation**
Computing pairwise IoU between `N` predicted boxes and `M` ground-truth boxes
(giving an `N×M` matrix) is needed for:
- NMS (self-overlap between predictions)
- mAP evaluation (matching predictions to ground truth)
- Anchor assignment during training

In [ ]:
def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """
    Computes pairwise IoU between two sets of boxes (vectorised).

    Args:
        boxes1 : (num_boxes1, 4) in [x1, y1, x2, y2] format
        boxes2 : (num_boxes2, 4) in [x1, y1, x2, y2] format

    Returns:
        iou : (num_boxes1, num_boxes2) pairwise IoU matrix
    """
    # Broadcast: (num_boxes1, 1, 4) and (1, num_boxes2, 4) → (num_boxes1, num_boxes2, 4)
    # Intersection: max of top-lefts, min of bottom-rights
    inter_x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])  # (N, M)
    inter_y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])  # (N, M)
    inter_x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])  # (N, M)
    inter_y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])  # (N, M)

    # Clamp at 0: negative width/height means no intersection
    inter_w = (inter_x2 - inter_x1).clamp(min=0)  # (N, M)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)  # (N, M)

    # (N, M)
    inter_area = inter_w * inter_h

    # Individual areas — broadcast to (N, M)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])  # (N,)
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])  # (M,)

    # Union = sum of areas - intersection to avoid double-counting
    # (N, M)
    union_area = area1[:, None] + area2[None, :] - inter_area

    return inter_area / (union_area + 1e-8)


# ── Verification ──────────────────────────────────────────────────────────────
b1 = torch.tensor([[0., 0., 10., 10.]])    # 10×10 box
b2 = torch.tensor([
    [0.,  0., 10., 10.],   # identical → IoU = 1.0
    [5.,  5., 15., 15.],   # 50% overlap
    [20., 20., 30., 30.],  # no overlap → IoU = 0.0
])

iou_mat = box_iou(b1, b2)
print("IoU matrix (1×3):")
print(iou_mat)
assert torch.isclose(iou_mat[0, 0], torch.tensor(1.0)), "Identical boxes must give IoU=1"
assert torch.isclose(iou_mat[0, 2], torch.tensor(0.0)), "Non-overlapping boxes must give IoU=0"
print("Assertions passed.")

# 3) Non-Maximum Suppression (NMS)

After a detector generates hundreds of candidate boxes, NMS retains only the
most confident, non-overlapping detection per object.

**Algorithm**
```
Input : boxes (N, 4), scores (N,), iou_threshold
Output: indices of kept boxes

1. Sort boxes by score (highest first)
2. Take the top-scoring box — keep it
3. Remove all remaining boxes with IoU > threshold against the kept box
4. Repeat from step 2 on remaining boxes
```

**Intuition**
Multiple nearby boxes likely all detected the *same* object.
The highest-scoring one is probably the most accurate localisation.
NMS discards the others.

**IoU threshold trade-off**
- Low threshold (0.3): aggressive suppression — might merge nearby objects
- High threshold (0.7): lenient — keeps more duplicates

In [ ]:
def nms(
    boxes: torch.Tensor,
    scores: torch.Tensor,
    iou_threshold: float = 0.5,
) -> torch.Tensor:
    """
    Greedy NMS: iteratively keeps the highest-scoring non-overlapping box.

    Args:
        boxes         : (num_boxes, 4) in [x1, y1, x2, y2] format
        scores        : (num_boxes,) confidence scores
        iou_threshold : Overlap threshold above which boxes are suppressed

    Returns:
        keep : 1-D tensor of indices of kept boxes, ordered by score
    """
    if boxes.numel() == 0:
        return torch.empty(0, dtype=torch.long)

    # Sort by score descending — best detections first
    # (num_boxes,) → (num_boxes,)
    order = scores.argsort(descending=True)

    keep = []

    while order.numel() > 0:
        # The current best box
        best_idx = order[0].item()
        keep.append(best_idx)

        if order.numel() == 1:
            break

        # Compare best box against all remaining boxes
        # (1, 4) vs (num_remaining, 4) → (1, num_remaining) → (num_remaining,)
        iou_with_best = box_iou(
            boxes[order[0:1]],  # (1, 4)
            boxes[order[1:]],   # (num_remaining, 4)
        ).squeeze(0)  # (num_remaining,)

        # Keep only boxes with IoU below the threshold (not too similar to best)
        # (num_remaining,) → indices into order[1:]
        low_overlap_mask = iou_with_best <= iou_threshold
        order            = order[1:][low_overlap_mask]

    return torch.tensor(keep, dtype=torch.long)


# ── Visualise NMS ──────────────────────────────────────────────────────────────
raw_boxes = torch.tensor([
    [10., 10., 80., 80.],
    [15., 12., 82., 78.],   # overlaps heavily with box 0
    [12., 11., 79., 81.],   # overlaps heavily with box 0
    [200., 200., 280., 270.],  # separate object
    [205., 198., 275., 268.],  # overlaps heavily with box 3
])
raw_scores = torch.tensor([0.95, 0.80, 0.70, 0.90, 0.60])

keep_idx = nms(raw_boxes, raw_scores, iou_threshold=0.5)
print(f"Raw  boxes : {len(raw_boxes)}")
print(f"After NMS  : {len(keep_idx)}  (indices: {keep_idx.tolist()})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colours = ["red", "orange", "yellow", "blue", "purple"]

for ax, (title, idx) in zip(
    axes,
    [("Before NMS", range(len(raw_boxes))), ("After NMS", keep_idx.tolist())]
):
    ax.set_xlim(0, 320); ax.set_ylim(0, 320)
    ax.invert_yaxis()
    ax.set_title(title)
    for i in idx:
        x1, y1, x2, y2 = raw_boxes[i].tolist()
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=colours[i], facecolor="none",
            label=f"s={raw_scores[i]:.2f}"
        )
        ax.add_patch(rect)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# 4) Anchor Boxes and YOLO Grid Encoding

**The anchor box idea** (Redmon et al., YOLOv2)

Instead of predicting absolute bounding box coordinates, a detector predicts
*offsets* relative to predefined anchor boxes. This makes the regression task
much easier because the anchors are pre-set to typical object sizes.

**YOLO grid**
The input image is divided into an `S × S` grid (e.g., 19×19).
Each cell predicts `A` boxes (one per anchor), each box consisting of:

```
(tx, ty)  — offset of box centre relative to cell top-left (sigmoid → [0,1])
(tw, th)  — log-scale offset relative to anchor width/height
p_obj     — objectness confidence (sigmoid → [0,1])
p_class   — class probabilities (softmax → [0,1], sums to 1)
```

**Decoding the prediction**

$$b_x = \sigma(t_x) + c_x \qquad b_y = \sigma(t_y) + c_y$$

$$b_w = p_w \cdot e^{t_w} \qquad b_h = p_h \cdot e^{t_h}$$

where $(c_x, c_y)$ is the cell offset and $(p_w, p_h)$ is the anchor size.

In [ ]:
def decode_yolo_output(
    raw_pred: torch.Tensor,
    anchors: torch.Tensor,
    grid_size: int,
    num_classes: int,
    img_size: int = 416,
) -> dict:
    """
    Decodes raw YOLO network output into interpretable bounding boxes.

    Args:
        raw_pred   : (batch_num, num_anchors * (5 + num_classes), grid_size, grid_size)
                     Raw output from the detection head
        anchors    : (num_anchors, 2)  [anchor_w, anchor_h] in pixels
        grid_size  : Number of grid cells per spatial dimension
        num_classes: Number of object classes
        img_size   : Input image size (assumed square)

    Returns:
        boxes      : (batch_num, grid_size*grid_size*num_anchors, 4) decoded [cx,cy,w,h]
        obj_conf   : (batch_num, grid_size*grid_size*num_anchors) objectness score
        cls_probs  : (batch_num, grid_size*grid_size*num_anchors, num_classes)
    """
    batch_num  = raw_pred.size(0)
    num_anchors = anchors.size(0)
    stride      = img_size // grid_size  # pixels per cell

    # Reshape: (batch_num, A*(5+C), G, G) → (batch_num, A, 5+C, G, G)
    # → (batch_num, A, G, G, 5+C)
    pred = raw_pred.view(batch_num, num_anchors, 5 + num_classes, grid_size, grid_size)
    pred = pred.permute(0, 1, 3, 4, 2)  # (batch_num, num_anchors, grid_size, grid_size, 5+C)

    # Build cell offset grid: cx, cy offsets for each grid cell
    # (grid_size,) → (1, 1, grid_size, grid_size)
    grid_x = torch.arange(grid_size, dtype=torch.float32).view(1, 1, 1, grid_size)
    grid_y = torch.arange(grid_size, dtype=torch.float32).view(1, 1, grid_size, 1)

    # Decode tx, ty → box centre in grid units, then scale to pixel space
    # σ(tx) + cx gives centre relative to grid; multiply by stride for pixels
    # (batch_num, num_anchors, grid_size, grid_size)
    bx = (torch.sigmoid(pred[..., 0]) + grid_x) * stride
    by = (torch.sigmoid(pred[..., 1]) + grid_y) * stride

    # Decode tw, th → box dimensions in pixels
    # anchor_w * exp(tw) gives absolute width
    # anchors: (num_anchors, 2) → (1, num_anchors, 1, 1)
    aw = anchors[:, 0].view(1, num_anchors, 1, 1)
    ah = anchors[:, 1].view(1, num_anchors, 1, 1)
    # (batch_num, num_anchors, grid_size, grid_size)
    bw = aw * torch.exp(pred[..., 2])
    bh = ah * torch.exp(pred[..., 3])

    # Objectness and class probabilities
    # (batch_num, num_anchors, grid_size, grid_size)
    obj_conf = torch.sigmoid(pred[..., 4])
    # (batch_num, num_anchors, grid_size, grid_size, num_classes)
    cls_probs = torch.softmax(pred[..., 5:], dim=-1)

    # Flatten grid and anchor dimensions for convenience
    # (batch_num, num_anchors*grid_size*grid_size, 4)
    boxes = torch.stack([bx, by, bw, bh], dim=-1).view(batch_num, -1, 4)
    # (batch_num, num_anchors*grid_size*grid_size)
    obj_conf  = obj_conf.view(batch_num, -1)
    # (batch_num, num_anchors*grid_size*grid_size, num_classes)
    cls_probs = cls_probs.view(batch_num, -1, num_classes)

    return {"boxes": boxes, "obj_conf": obj_conf, "cls_probs": cls_probs}


# ── Dry run with synthetic output ─────────────────────────────────────────────
GRID_SIZE   = 13
NUM_ANCHORS = 3
NUM_CLASSES = 80
IMG_SIZE    = 416

anchors = torch.tensor([
    [116., 90.],   # large objects
    [30.,  61.],   # medium
    [10.,  13.],   # small
])

# Synthetic raw prediction tensor
raw = torch.randn(2, NUM_ANCHORS * (5 + NUM_CLASSES), GRID_SIZE, GRID_SIZE)

decoded = decode_yolo_output(raw, anchors, GRID_SIZE, NUM_CLASSES, IMG_SIZE)

print(f"Raw prediction : {raw.shape}")                       # (2, 255, 13, 13)
print(f"Decoded boxes  : {decoded['boxes'].shape}")          # (2, 507, 4)
print(f"Obj confidence : {decoded['obj_conf'].shape}")       # (2, 507)
print(f"Class probs    : {decoded['cls_probs'].shape}")      # (2, 507, 80)
print(f"Total predictions per image: {NUM_ANCHORS} × {GRID_SIZE}² = {NUM_ANCHORS*GRID_SIZE**2}")

# 5) Full Detection Pipeline

Combining decoding, score thresholding, and NMS into one pipeline.

**Score thresholding**
The final class score for each box = objectness × class_probability.
This filters out predictions where the network is uncertain about either
the presence or identity of an object.

In [ ]:
def detect(
    raw_pred: torch.Tensor,
    anchors: torch.Tensor,
    grid_size: int,
    num_classes: int,
    score_threshold: float = 0.25,
    iou_threshold: float   = 0.45,
    img_size: int          = 416,
) -> list:
    """
    Full detection pipeline: decode → threshold → NMS.

    Args:
        raw_pred        : Raw network output
        anchors         : Anchor box dimensions
        grid_size       : Grid resolution
        num_classes     : Number of object classes
        score_threshold : Minimum confidence score to keep a box
        iou_threshold   : NMS IoU threshold
        img_size        : Input image size

    Returns:
        detections : List of dicts per image:
                     {'boxes': (K,4), 'scores': (K,), 'labels': (K,)}
    """
    decoded   = decode_yolo_output(raw_pred, anchors, grid_size, num_classes, img_size)
    batch_num = raw_pred.size(0)
    results   = []

    for i in range(batch_num):
        # Final detection score = objectness × max class probability
        # (num_preds, num_classes) → (num_preds,)
        class_scores, class_labels = decoded["cls_probs"][i].max(dim=1)
        # (num_preds,)
        scores = decoded["obj_conf"][i] * class_scores

        # Threshold: keep only high-confidence predictions
        mask    = scores > score_threshold
        boxes_f = cxcywh_to_xyxy(decoded["boxes"][i][mask])
        scores_f = scores[mask]
        labels_f = class_labels[mask]

        if boxes_f.numel() == 0:
            results.append({"boxes": torch.empty(0,4), "scores": torch.empty(0), "labels": torch.empty(0)})
            continue

        # Apply NMS per class to avoid suppressing different classes
        keep_all = []
        for cls_id in labels_f.unique():
            cls_mask  = labels_f == cls_id
            keep_cls  = nms(boxes_f[cls_mask], scores_f[cls_mask], iou_threshold)
            # Map back to full-set indices
            full_idx  = cls_mask.nonzero(as_tuple=True)[0]
            keep_all.append(full_idx[keep_cls])

        keep = torch.cat(keep_all)
        results.append({
            "boxes":  boxes_f[keep],
            "scores": scores_f[keep],
            "labels": labels_f[keep],
        })

    return results


# ── Test the pipeline ──────────────────────────────────────────────────────────
detections = detect(raw, anchors, GRID_SIZE, NUM_CLASSES)
for i, det in enumerate(detections):
    print(f"Image {i}: {len(det['boxes'])} detections after thresholding + NMS")

# 6) Modern Inference with YOLOv8

The previous sections implement the core detection maths manually.
In practice, researchers use `ultralytics` (YOLOv8) which encapsulates
everything above into a production-grade system.

**What YOLOv8 adds on top of what we implemented**
- Anchor-free design (predicts box centres directly — simpler training)
- CSP (Cross-Stage Partial) backbone
- CIOU loss (better bounding box regression than MSE)
- Data augmentation mosaic and mixup during training
- ONNX/TensorRT export for deployment

The 3-line inference API makes the contrast clear: the concepts you
just implemented are all there, just highly optimised and abstracted.

In [ ]:
# Install ultralytics (YOLOv8 package)
# !pip install ultralytics --quiet

# NOTE: The cell below requires ultralytics installed.
# Uncomment and run in Colab.

# from ultralytics import YOLO
# from PIL import Image
# import requests
# from io import BytesIO
#
# Load pretrained YOLOv8n (nano — fastest, smallest)
# model_yolo = YOLO("yolov8n.pt")
#
# Run inference on a URL image
# results = model_yolo("https://ultralytics.com/images/bus.jpg")
#
# Access results
# for r in results:
#     print(f"Detected {len(r.boxes)} objects")
#     for box in r.boxes:
#         # box.xyxy  — (1, 4) corner format
#         # box.conf  — confidence score
#         # box.cls   — class index
#         cls_name = model_yolo.names[int(box.cls)]
#         print(f"  {cls_name}: conf={float(box.conf):.3f}  box={box.xyxy.tolist()}")
#
# Show annotated image
# results[0].show()

print("YOLOv8 pseudocode shown — install ultralytics and uncomment to run.")
print()
print("Three-line summary:")
print("  1. model = YOLO('yolov8n.pt')")
print("  2. results = model(image_path)")
print("  3. for r in results: process r.boxes")

# ── YOLO vs what we built — correspondence table ───────────────────────────────
table = {
    "IoU computation"   : "box_iou()   →  torchvision.ops.box_iou",
    "NMS"               : "nms()       →  torchvision.ops.nms (CUDA optimised)",
    "Box decoding"      : "decode_yolo_output()  →  model.postprocess()",
    "Score thresholding": "scores > threshold    →  conf= argument to model()",
    "Anchor assignment" : "manual loops → TAL (Task-Aligned Assigner in YOLOv8)",
}
print("\nOur implementation  →  YOLOv8 equivalent")
print("-" * 55)
for k, v in table.items():
    print(f"  {k:<22} : {v}")

# Summary

| Concept | Formula | Purpose |
|---|---|---|
| IoU | $ \frac{|A \cap B|}{|A \cup B|}$ | Measure overlap; backbone of all detection metrics |
| NMS | Keep highest-score box, suppress overlapping | Collapse duplicate detections |
| YOLO grid | $b_x = \sigma(t_x) + c_x$ | Predict offsets, not absolute coordinates |
| Score threshold | $s = p_{obj} \times p_{class}$ | Remove low-confidence predictions before NMS |

**Key research connections**
- **mAP (mean Average Precision)** is computed by sweeping the score threshold
  and computing area under the precision-recall curve, using IoU > 0.5 (or 0.5:0.95)
  to determine true positives — built on `box_iou`.
- **Anchor-free detectors** (FCOS, CenterNet, YOLOv8) avoid anchors entirely,
  directly predicting the distance from each point to the box edges.
- **DETR** replaces NMS with set-prediction via Hungarian matching — the attention
  mechanism naturally handles the suppression problem.